# T5

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

import json
from typing import List, Dict
import numpy as np
import sacrebleu
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)


def load_jsonl(path: str) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def build_dataset(train_path: str, valid_path: str):
    train_rows = load_jsonl(train_path)
    valid_rows = load_jsonl(valid_path)
    return Dataset.from_list(train_rows), Dataset.from_list(valid_rows)

def compute_sacrebleu(preds: List[str], refs: List[str]) -> float:
    bleu = sacrebleu.corpus_bleu([p.strip() for p in preds], [[r.strip() for r in refs]])
    return float(bleu.score)

def finetune_mt5_zh_en(
    train_path="data/train_10k.jsonl",
    valid_path="data/valid.jsonl",
    src_key="zh",
    tgt_key="en",
    model_name="Helsinki-NLP/opus-mt-zh-en",
    out_dir="./opus_zh_en_ckpt",
    max_src_len=96,
    max_tgt_len=64,
    lr=1e-4,
    batch_size=8,
    grad_accum=16,
    epochs=5,
    beam_size=1,
):
    train_ds, valid_ds = build_dataset(train_path, valid_path)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, use_safetensors=True,)

    def preprocess(batch):
        inputs = batch[src_key]
        targets = batch[tgt_key]

        model_inputs = tokenizer(
            inputs,
            max_length=max_src_len,
            truncation=True,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=max_tgt_len,
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
    valid_tok = valid_ds.map(preprocess, batched=True, remove_columns=valid_ds.column_names)

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )

    def compute_metrics(eval_pred):
        preds, labels = eval_pred

        if isinstance(preds, tuple):
            preds = preds[0]

        preds = np.asarray(preds)
        labels = np.asarray(labels)

        if preds.ndim == 3:
            preds = preds.argmax(axis=-1)

        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        preds = [list(map(int, row)) for row in preds]
        labels = [list(map(int, row)) for row in labels]

        pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
        ref_text  = tokenizer.batch_decode(labels, skip_special_tokens=True)

        return {"bleu": compute_sacrebleu(pred_text, ref_text)}


    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=50,
        report_to="none",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=beam_size,
        gradient_accumulation_steps=grad_accum,
        fp16=True,
        predict_with_generate=False,
        dataloader_num_workers=0,
        remove_unused_columns=False,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    '''
    ex = train_tok[0] if hasattr(train_tok, "__getitem__") else train_ds[0]
    print("keys:", ex.keys())

    labels = ex["labels"]
    print("labels type:", type(labels))
    print("labels len:", len(labels))
    print("labels head:", labels[:30])

    neg100 = sum(1 for x in labels if x == -100)
    print("neg100:", neg100, "/", len(labels), "ratio=", neg100/len(labels))

    valid = sum(1 for x in labels if x != -100 and x != tokenizer.pad_token_id)
    print("valid label tokens:", valid)

    tmp = data_collator([train_tok[i] for i in range(8)])
    lb = tmp["labels"]
    print("batch labels shape:", lb.shape)
    print("batch labels valid ratio:", (lb != -100).float().mean().item())
    print("loss test:", model(input_ids=tmp["input_ids"].to(model.device),
                            attention_mask=tmp["attention_mask"].to(model.device),
                            labels=tmp["labels"].to(model.device)).loss.item())

    print("label unique sample:", torch.unique(tmp["labels"][0])[:20])
    print("pad_id:", tokenizer.pad_token_id)
    '''


    batch = next(iter(torch.utils.data.DataLoader(
    train_tok, batch_size=2, shuffle=False,
    collate_fn=data_collator
    )))
    batch = {k: v.to(model.device) for k, v in batch.items()}

    '''
    out = model(**batch)
    print("loss:", out.loss)
    print("loss is nan:", torch.isnan(out.loss).item())

    logits = out.logits
    print("logits nan:", torch.isnan(logits).any().item(), "inf:", torch.isinf(logits).any().item())
    '''

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
#        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    print("Final eval:", metrics)

    trainer.save_model(out_dir + "/best")
    tokenizer.save_pretrained(out_dir + "/best")
    return model, tokenizer, metrics


c:\Users\250010155\AppData\Local\miniconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mt5_model, mt5_tok, mt5_metrics = finetune_mt5_zh_en(
    train_path="data/train_10k.jsonl",
    valid_path="data/valid.jsonl",
    epochs=5,
    lr=3e-4,
    batch_size=8,
    grad_accum=16,
    beam_size=1
)

c:\Users\250010155\AppData\Local\miniconda3\envs\py310\lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Map: 100%|██████████| 500/500 [00:00<00:00, 5038.81 examples/s]
C:\Users\250010155\AppData\Local\Temp\ipykernel_39768\2399208365.py:169: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


loss: tensor(0.9814, grad_fn=<NllLossBackward0>)
loss is nan: False
logits nan: False inf: False


Epoch,Training Loss,Validation Loss
1,1.721800,2.171935
2,1.113100,2.335079
3,0.814300,2.442576
4,0.543000,2.542295
5,0.452900,2.601238


c:\Users\250010155\AppData\Local\miniconda3\envs\py310\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[65000]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Final eval: {'eval_loss': 2.6012377738952637, 'eval_runtime': 5.8682, 'eval_samples_per_second': 85.205, 'eval_steps_per_second': 85.205, 'epoch': 5.0}


In [3]:
import torch
mt5_model.eval()


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(65001, 512, padding_idx=65000)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(65001, 512, padding_idx=65000)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [5]:
import json
from tqdm import tqdm
import sacrebleu

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

valid_rows = load_jsonl("data/valid.jsonl")
src_texts = [r["zh"] for r in valid_rows]
ref_texts = [r["en"] for r in valid_rows]

pred_texts = []
batch_size = 4  # RTX A1000 很安全

for i in tqdm(range(0, len(src_texts), batch_size)):
    batch_src = src_texts[i:i+batch_size]

    inputs = mt5_tok(
        ["translate Chinese to English: " + s for s in batch_src],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=96,
    ).to(mt5_model.device)

    with torch.no_grad():
        outputs = mt5_model.generate(
            **inputs,
            num_beams=1,
            max_length=64,
        )

    preds = mt5_tok.batch_decode(outputs, skip_special_tokens=True)
    pred_texts.extend(preds)
    
bleu = sacrebleu.corpus_bleu(pred_texts, [ref_texts])
print("Validation BLEU:", bleu.score)

100%|██████████| 125/125 [00:40<00:00,  3.07it/s]

Validation BLEU: 15.004835253208794
